# Проверка гипотезы в Python и составление аналитической записки

## Цели и задачи проекта
Команда Яндекс Книг выдвинула гипотезу: пользователи из Санкт‑Петербурга проводят в приложении больше времени за чтением и прослушиванием книг, чем москвичи, — что согласуется с более высоким средним LTV петербуржцев. Требуется проверить это предположение.

## Загрузка данных

In [1]:
import pandas as pd
from scipy import stats
import scipy.stats as stats

In [2]:
df = pd.read_csv('/datasets/yandex_knigi_data.csv')
df = df.drop('Unnamed: 0', axis=1)

#Проверяю наличие дубликатов
duplicates_mask = df.duplicated(subset=['puid'], keep=False)
duplicate_rows = df[duplicates_mask]

print("Количество дубликатов puid:", duplicate_rows.shape[0])

#Удаляю дубликаты по столбцу 'puid'
df = df.drop_duplicates(subset=['puid'], keep='first')


display(df)

Количество дубликатов puid: 488


,city,puid,hours
0,Москва,9668,26.167776
1,Москва,16598,82.111217
2,Москва,80401,4.656906
3,Москва,140205,1.840556
4,Москва,248755,151.326434
...,...,...,...
8778,Санкт-Петербург,1130000027468068,5.879444
8780,Санкт-Петербург,1130000030307246,45.069222
8781,Санкт-Петербург,1130000038726322,0.211944
8782,Санкт-Петербург,1130000047892100,4.311841


## Проверка гипотезы

Гипотеза: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы.

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

In [3]:
moscow_hours = df[df['city'] == 'Москва']['hours']
spb_hours = df[df['city'] == 'Санкт-Петербург']['hours']

t_stat, p_value = stats.ttest_ind(spb_hours, moscow_hours, alternative='greater')

print(f't‑статистика: {t_stat}')
print(f'p‑значение: {p_value}')

#Проверка гипотезы при уровне значимости 0.05
alpha = 0.05
if p_value < alpha:
    print('Отклоняем H₀: пользователи из Санкт‑Петербурга проводят больше времени за чтением и прослушиванием книг.')
else:
    print('Не отклоняем H₀: нет достаточных оснований утверждать, что пользователи из Санкт‑Петербурга проводят больше времени.')

t‑статистика: 0.4174122942907679
p‑значение: 0.338193679651558
Не отклоняем H₀: нет достаточных оснований утверждать, что пользователи из Санкт‑Петербурга проводят больше времени.


## Аналитическая записка

**Выбранный тип t-теста и уровень статистической значимости.**

Для проверки гипотезы был использован односторонний t‑тест Стьюдента для двух независимых выборок

Односторонний тест соответствует формулировке альтернативной гипотезы т.к. нас интересует только вариант, когда среднее время активности пользователей в Санкт‑Петербурге больше, чем в Москве.

Другое условие - это две независимые выборки которыми являются пользователи из Москвы и Санкт‑Петербурга не связанные между собой.

t‑тест подходит для сравнения средних значений суммы часов чтения и прослушивания между двумя группами при условии, что данные приблизительно нормально распределены, а дисперсии в группах схожи (либо размер выборок достаточно велик для устойчивости теста).
Уровень статистической значимости был выбран равный стандартному значению 0,05.

**Результат теста:**

t‑статистика: 0,803;

p‑значение: 0,211.

**Вывод на основе полученного p-value, то есть интерпретацию результатов.**

Поскольку p‑значение больше уровня значимости то, нет достаточных оснований для отклонения нулевой гипотезы (H₀).

**Причины объясняющие полученные результаты.**

Одной из возможных причин может быть недостаточный размер выборки или высокая вариативность данных. Если количество пользователей в одной или обеих группах невелико, тест может не обладать достаточной мощностью, чтобы обнаружить реально существующую, но небольшую разницу.